# E: T4 first-round verification and answer repair

In Colab, choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
The same notebook can be imported on Kaggle: enable GPU and Internet, and add the experiment bundle as a private input.

Upload `answer_repair_v1_gpu.tar.gz`; it contains frozen code and 50 development samples each for FEVER and NQ.
First run the FEVER 10-sample engineering smoke, inspect all outputs, then run the optional development experiment.
Development samples exclude Diagnosis v1/v2 validation and test IDs. Smoke is a subset of the development set.
Reference answers are used only for offline scoring and are not passed to the repair model. A valid citation does not mean semantic support has been verified.


In [ ]:
import json, os, sys, subprocess, hashlib, tarfile, shutil
from pathlib import Path
import torch

assert sys.version_info >= (3, 11), "需要 Python 3.11+"
assert torch.cuda.is_available(), "请先选择 GPU 运行时；当前 PyTorch 看不到 CUDA"
print(json.dumps({"python": sys.version, "torch": torch.__version__,
                  "cuda": torch.version.cuda,
                  "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}, indent=2))
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
OUTPUT = WORK / 'answer_repair_v1_outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Reuse the file if it was already uploaded from the Colab left-hand file pane; otherwise show the upload dialog.
EXPECTED_BUNDLE_SHA256 = 'c6c9b0b748cc8d58a89a8f83a6f71ca61cf7e48527a8e8a0ce29ba5742898ee1'
archive = WORK / 'answer_repair_v1_gpu.tar.gz'
if not archive.exists() and Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').rglob('answer_repair_v1_gpu.tar.gz'))
    assert len(candidates) == 1, '请在 Kaggle 添加唯一的实验包输入'
    archive = candidates[0]
elif not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    assert 'answer_repair_v1_gpu.tar.gz' in uploaded, '请选择 answer_repair_v1_gpu.tar.gz'
    archive.write_bytes(uploaded['answer_repair_v1_gpu.tar.gz'])

assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256, '实验包哈希不一致'
extract_root = WORK / ('e_bundle_' + EXPECTED_BUNDLE_SHA256[:12])
BUNDLE = extract_root / 'answer_repair_v1_gpu'
if not BUNDLE.exists():
    extract_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as handle:
        handle.extractall(extract_root, filter='data')
checksums = json.loads((BUNDLE / 'bundle_checksums.json').read_text())
assert all(hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() == value
           for name, value in checksums.items()), '代码或数据文件校验失败'
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
print('包校验通过:', len(checksums), '个文件')
print(json.dumps({ds: {'development': item['development'], 'smoke': item['smoke']}
                  for ds, item in manifest['datasets'].items()}, indent=2))


In [ ]:
def run_logged(argv, log_name, cwd=BUNDLE):
    # Explicitly forward subprocess output so the notebook does not only show a running status.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    with (OUTPUT / log_name).open('a', encoding='utf-8') as log:
        log.write('COMMAND ' + json.dumps(argv) + '\n')
        log.flush()
        process = subprocess.Popen(argv, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, argv)

# Use the host CUDA PyTorch; the environment check records the versions after the actual install.
run_logged([sys.executable, '-m', 'pip', 'install', '-r',
            str(BUNDLE / 'requirements-answer-repair.txt')], 'install.log')
run_logged([sys.executable, 'scripts/check_answer_repair_environment.py',
            '--require-cuda', '--out', str(OUTPUT / 'environment.json')], 'environment.log')
run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], 'tests.log')
shutil.copy2(BUNDLE / 'manifest.json', OUTPUT / 'manifest.json')
shutil.copy2(BUNDLE / 'bundle_checksums.json', OUTPUT / 'bundle_checksums.json')


In [ ]:
SMOKE_DATASET = 'fever'
SMOKE_RESULT = OUTPUT / f'{SMOKE_DATASET}_smoke10_verify.jsonl'
SMOKE_ARGS = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', SMOKE_DATASET,
              '--c1', str(BUNDLE / f'inputs/{SMOKE_DATASET}_smoke10_c1.jsonl'),
              '--all', '--out', str(SMOKE_RESULT)]
run_logged(SMOKE_ARGS + ['--check-only'], 'smoke_preflight.log')


In [ ]:
# Real 7B NF4 calls; the first run downloads the model. Each sample is appended so the same environment can resume.
run_logged(SMOKE_ARGS, 'smoke_run.log')


In [ ]:
summary = json.loads(SMOKE_RESULT.with_suffix('.summary.json').read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))
source_rows = {str(row['sample_id']): row for row in
               map(json.loads, (BUNDLE / f'inputs/{SMOKE_DATASET}_smoke10_c1.jsonl').read_text().splitlines())}
results = list(map(json.loads, SMOKE_RESULT.read_text().splitlines()))
review = []
for result in results:
    source = source_rows[result['sample_id']]
    row = {**result, 'reference_answer_or_label': source['gold_answer_or_label'],
           'input_evidence': source['evidence'],
           'manual_review': {'supports_final_answer': None, 'notes': ''}}
    review.append(row)
    print(json.dumps({k: result.get(k) for k in
                     ('sample_id', 'status', 'initial_answer', 'final_answer', 'citations', 'validation_error')},
                    ensure_ascii=False))
review_file = OUTPUT / f'{SMOKE_DATASET}_smoke10_review.jsonl'
review_file.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in review))
print('离线人工审核文件:', review_file)


## Optional development experiment

First review JSON, citations, corrections, and regressions in the smoke run. After confirming execution is healthy, set
`RUN_DEVELOPMENT` to `True` in the next cell. It runs Verify and Reanswer
contrasts on 50 FEVER and 50 NQ samples, for at most 200 extra generations. These are development results, not a final independent test.


In [ ]:
RUN_DEVELOPMENT = False
if RUN_DEVELOPMENT:
    for dataset in ('fever', 'nq'):
        for strategy in ('verify_repair', 'reanswer'):
            out = OUTPUT / f'{dataset}_dev50_{strategy}.jsonl'
            run_logged([sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', dataset,
                        '--c1', str(BUNDLE / f'inputs/{dataset}_dev50_c1.jsonl'), '--all',
                        '--strategy', strategy, '--out', str(out)], f'{dataset}_{strategy}.log')
else:
    print('开发实验未启动；先审核 smoke。')


In [ ]:
# Export existing results and environment records; model weights are not packaged.
export = shutil.make_archive(str(WORK / 'answer_repair_v1_outputs'), 'zip', OUTPUT)
print('结果包:', export)
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(export)
